# Multi-Agent Fantasy Storytelling — V2
### Mistral-7B Fine-Tuning + Narrator Agent + Hot-Swappable LoRA

**What changed from V1:**
- Base model is **Mistral-7B-v0.1** (pretrained), not GPT-2 trained from scratch
- **Two-phase training**: general fantasy fine-tune → per-agent adapter fine-tune
- **Narrator agent** generates atmospheric prose to open and close each story beat
- **All 4 characters respond sequentially** per beat — each sees the previous line
- **Shared model** (one Mistral-7B + 5 hot-swapped adapters = ~15 GB VRAM, not 70 GB)
- **CharacterFactory** enforces exactly 4 characters from the user's premise

**Story beat flow:**
```
[USER]     premise / next scene
[NARRATOR] opens the scene with atmospheric prose
[CHAR_1]   responds (sees narrator opening)
[CHAR_2]   responds (sees char 1's line)
[CHAR_3]   responds (sees char 2's line)
[CHAR_4]   responds (sees char 3's line)
[NARRATOR] closes / transitions
```

## 0. Install Dependencies
Run once, comment out after.

In [ ]:
# !pip install 'transformers>=4.36' 'peft>=0.7' accelerate datasets bitsandbytes trl
# !pip install faiss-cpu sentence-transformers
# !pip install torch torchvision torchaudio
# !pip install huggingface_hub requests
# !pip install flash-attn --no-build-isolation  # strongly recommended for H100

## 1. Imports & Device Setup

In [1]:
import os, json, random, re, glob
import numpy as np
import torch
import time as _time
import requests
from pathlib import Path
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    pipeline,
)
from datasets import load_dataset
from huggingface_hub import snapshot_download
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    PeftModel,
)
from sentence_transformers import SentenceTransformer
import faiss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

/home/rja3097/.conda/envs/cuda_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
GPU: NVIDIA A100-SXM4-80GB
Memory: 85.1 GB


## 2. Configuration

In [2]:
# ── Paths ─────────────────────────────────────────────────────────────
BASE_DIR  = Path('./fantasy_v2')
DATA_DIR  = BASE_DIR / 'data'
MODEL_DIR = BASE_DIR / 'models'
LORA_DIR  = BASE_DIR / 'lora_adapters'
RAG_DIR   = BASE_DIR / 'rag'
for d in [DATA_DIR, MODEL_DIR, LORA_DIR, RAG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Base Model ────────────────────────────────────────────────────────
BASE_MODEL_ID = 'mistralai/Mistral-7B-v0.1'
MAX_SEQ_LEN   = 1024

# ── Phase 1: General Fantasy Fine-Tuning ──────────────────────────────
P1_BATCH_SIZE  = 4
P1_GRAD_ACCUM  = 8      # effective batch = 32
P1_LR          = 2e-4
P1_EPOCHS      = 2
P1_LORA_R      = 16
P1_LORA_ALPHA  = 32

# ── Phase 2: Per-Agent Adapter Fine-Tuning ────────────────────────────
P2_BATCH_SIZE   = 4
P2_GRAD_ACCUM   = 4     # effective batch = 16
P2_LR           = 1e-4
P2_EPOCHS       = 3
P2_LORA_R       = 64
P2_LORA_ALPHA   = 128
P2_LORA_DROPOUT = 0.05
P2_N_SAMPLES    = 500

NUM_CHARACTERS = 4  # exactly 4 character agents; narrator is the 5th

print(f'Base model : {BASE_MODEL_ID}')
print(f'Context    : {MAX_SEQ_LEN} tokens')
print(f'Agents     : {NUM_CHARACTERS} characters + 1 narrator')

Base model : mistralai/Mistral-7B-v0.1
Context    : 1024 tokens
Agents     : 4 characters + 1 narrator


## 3. Dataset Loading

Four sources — LIGHT (fantasy dialogue), Character-LLM (character voices),
WritingPrompts (narrative prose), Project Gutenberg (public-domain fantasy).

### 3a. LIGHT Dataset

In [3]:
def load_light_dataset():
    try:
        ds = load_dataset('npc-engine/light-batch-summarize-dialogue')
        print(f'LIGHT loaded. Splits: {list(ds.keys())}')
        return ds
    except Exception as e:
        print(f'Could not load LIGHT: {e}')
        return None

def extract_light_dialogues(ds):
    texts = []
    for row in ds.get('train', []):
        raw = (row.get('dialogue_text') or '').strip()
        if not raw:
            continue
        block = '[SETTING] fantasy encounter\n'
        for line in raw.split('\n'):
            if ':' in line:
                speaker, _, utterance = line.partition(':')
                tag = speaker.strip().upper().replace(' ', '_')
                block += f'[{tag}] {utterance.strip()}\n'
            else:
                block += line + '\n'
        texts.append(block.strip())
    print(f'Extracted {len(texts)} LIGHT blocks')
    return texts

light_ds    = load_light_dataset()
light_texts = extract_light_dialogues(light_ds) if light_ds else []

Generating test split: 100%|██████████| 2000/2000 [00:00<00:00, 505916.89 examples/s]


LIGHT loaded. Splits: ['train', 'validation', 'test']
Extracted 17075 LIGHT blocks


### 3b. Character-LLM Dataset

In [4]:
def load_character_llm_dataset():
    try:
        local_dir = DATA_DIR / 'character_llm_raw'
        print('Downloading fnlp/character-llm-data...')
        snapshot_download(
            repo_id='fnlp/character-llm-data',
            repo_type='dataset',
            local_dir=str(local_dir),
        )
        return local_dir
    except Exception as e:
        print(f'Could not download Character-LLM: {e}')
        return None

def extract_character_llm_texts(local_dir):
    if local_dir is None:
        return []
    texts = []
    prompted_files = glob.glob(str(local_dir / 'prompted' / '*.jsonl'))
    for fpath in prompted_files:
        char_name = Path(fpath).stem.replace('prompted_agent_dialogue_', '')
        tag = char_name.upper().replace(' ', '_')
        with open(fpath, encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    row = json.loads(line)
                except json.JSONDecodeError:
                    continue
                if 'prompt' in row and 'output' in row:
                    block = (f'[CHARACTER] {char_name}\n'
                             f'[USER] {row["prompt"].strip()}\n'
                             f'[{tag}] {row["output"].strip()}')
                elif 'dialogue' in row:
                    setting = row.get('setting') or row.get('location') or 'unknown'
                    block = f'[CHARACTER] {char_name}\n[SETTING] {setting}\n'
                    for turn in (row.get('dialogue') or []):
                        speaker = turn.get('role', 'speaker')
                        text    = turn.get('content', turn.get('text', ''))
                        t_tag   = '[USER]' if speaker == 'human' else f'[{tag}]'
                        block  += f'{t_tag} {text}\n'
                    block = block.strip()
                else:
                    continue
                texts.append(block)
    print(f'Extracted {len(texts)} Character-LLM blocks from {len(prompted_files)} files')
    return texts

char_local  = load_character_llm_dataset()
char_texts  = extract_character_llm_texts(char_local)

Fetching 29 files: 100%|██████████| 29/29 [00:00<00:00, 634.31it/s]


Extracted 14174 Character-LLM blocks from 9 files


### 3c. WritingPrompts Dataset (Fantasy-Filtered Narrative Prose)

In [5]:
FANTASY_KEYWORDS = {
    'wizard', 'dragon', 'magic', 'kingdom', 'quest', 'sword', 'elf', 'dwarf',
    'spell', 'sorcerer', 'dungeon', 'prophecy', 'enchant', 'knight', 'castle',
    'realm', 'potion', 'beast', 'hero', 'ghost', 'demon', 'mystical', 'curse',
    'rune', 'tome', 'ancient', 'dark', 'battle', 'throne', 'warrior', 'mage',
}

def load_writing_prompts(max_samples=20000):
    try:
        ds = load_dataset('euclaise/writingprompts', split='train')
        print(f'WritingPrompts loaded: {len(ds)} total')
    except Exception as e:
        print(f'Could not load WritingPrompts: {e}')
        return []
    texts = []
    for row in ds:
        prompt = (row.get('prompt') or row.get('source') or '').strip()
        story  = (row.get('story')  or row.get('target') or row.get('text') or '').strip()
        if not story or len(story) < 150:
            continue
        combined = (prompt + ' ' + story[:500]).lower()
        if not any(kw in combined for kw in FANTASY_KEYWORDS):
            continue
        texts.append(f'[SETTING] {prompt}\n[NARRATOR] {story[:2000]}')
        if len(texts) >= max_samples:
            break
    print(f'Extracted {len(texts)} fantasy WritingPrompts blocks')
    return texts

wp_texts = load_writing_prompts()

Generating validation split: 100%|██████████| 15620/15620 [00:00<00:00, 95654.39 examples/s]


WritingPrompts loaded: 272600 total
Extracted 20000 fantasy WritingPrompts blocks


### 3d. Project Gutenberg — Public Domain Fantasy Prose

In [6]:
GUTENBERG_BOOKS = [
    (62,   'A Princess of Mars — Burroughs'),
    (170,  'The Wood Beyond the World — Morris'),
    (1251, 'Le Morte d\'Arthur — Malory'),
    (5670, 'The Story of the Glittering Plain — Morris'),
    (9782, 'Champions of the Round Table — Pyle'),
    (20776,'Beowulf — Anonymous'),
]

def _fetch_gutenberg_book(book_id):
    url = f'https://www.gutenberg.org/cache/epub/{book_id}/pg{book_id}.txt'
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    text = r.text
    for marker in ['*** START OF THE PROJECT GUTENBERG', '***START OF THE PROJECT GUTENBERG']:
        idx = text.find(marker)
        if idx != -1:
            text = text[text.index('\n', idx) + 1:]
            break
    for marker in ['*** END OF THE PROJECT GUTENBERG', '***END OF THE PROJECT GUTENBERG']:
        idx = text.find(marker)
        if idx != -1:
            text = text[:idx]
            break
    return text.strip()

def _chunk_book(text, chunk_chars=1500, overlap=150):
    chunks, i = [], 0
    while i < len(text):
        chunk = text[i: i + chunk_chars].strip()
        if len(chunk) > 200:
            chunks.append(chunk)
        i += chunk_chars - overlap
    return chunks

def load_gutenberg_texts():
    all_chunks = []
    for book_id, title in GUTENBERG_BOOKS:
        print(f'Fetching: {title} ...', end=' ', flush=True)
        try:
            raw    = _fetch_gutenberg_book(book_id)
            chunks = _chunk_book(raw)
            all_chunks.extend(f'[NARRATOR] {c}' for c in chunks)
            print(f'{len(chunks)} chunks')
        except Exception as e:
            print(f'FAILED ({e})')
        _time.sleep(1)
    print(f'\nTotal Gutenberg chunks: {len(all_chunks)}')
    return all_chunks

gutenberg_texts = load_gutenberg_texts()

Fetching: A Princess of Mars — Burroughs ... 280 chunks
Fetching: The Wood Beyond the World — Morris ... 266 chunks
Fetching: Le Morte d'Arthur — Malory ... 664 chunks
Fetching: The Story of the Glittering Plain — Morris ... 238 chunks
Fetching: Champions of the Round Table — Pyle ... 514 chunks
Fetching: Beowulf — Anonymous ... 604 chunks

Total Gutenberg chunks: 2566


### 3e. Combine & Save

In [7]:
all_texts = light_texts + char_texts + wp_texts + gutenberg_texts
random.shuffle(all_texts)

if not all_texts:
    print('No data loaded — injecting synthetic fallback')
    all_texts = [
        '[SETTING] A dark dungeon.\n[NARRATOR] Torchlight flickered against stone walls slick with moisture.',
        '[CHARACTER] Wizard\n[PERSONA] Ancient spellcaster.\n[USER] What spell will you cast?\n[WIZARD] None that you could name.',
    ] * 100

print(f'Total examples : {len(all_texts)}')
print(f'  LIGHT         : {len(light_texts)}')
print(f'  Character-LLM : {len(char_texts)}')
print(f'  WritingPrompts: {len(wp_texts)}')
print(f'  Gutenberg     : {len(gutenberg_texts)}')

corpus_path = DATA_DIR / 'combined_corpus.txt'
with open(corpus_path, 'w', encoding='utf-8') as f:
    f.write('\n\n'.join(all_texts))
print(f'Corpus saved to {corpus_path}')

Total examples : 53815
  LIGHT         : 17075
  Character-LLM : 14174
  WritingPrompts: 20000
  Gutenberg     : 2566
Corpus saved to fantasy_v2/data/combined_corpus.txt


## 4. Tokenizer & Dataset Preparation

In [8]:
print(f'Loading tokenizer from {BASE_MODEL_ID}...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'  # right-pad for causal LM training

special_tokens = {
    'additional_special_tokens': [
        '[SETTING]', '[CHARACTER]', '[PERSONA]', '[USER]',
        '[NARRATOR]', '[LORE]', '[STORY SO FAR]',
    ]
}
tokenizer.add_special_tokens(special_tokens)
print(f'Vocab size (with special tokens): {len(tokenizer)}')


class FantasyTextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=MAX_SEQ_LEN):
        self.examples = []
        for text in texts:
            enc = tokenizer(
                text,
                truncation=True,
                max_length=max_length,
                padding='max_length',
                return_tensors='pt',
            )
            self.examples.append({
                'input_ids':      enc['input_ids'].squeeze(),
                'attention_mask': enc['attention_mask'].squeeze(),
                'labels':         enc['input_ids'].squeeze().clone(),
            })

    def __len__(self):              return len(self.examples)
    def __getitem__(self, idx):     return self.examples[idx]


split_idx     = int(0.9 * len(all_texts))
train_texts   = all_texts[:split_idx]
val_texts     = all_texts[split_idx:]

print('Tokenizing training set...')
train_dataset = FantasyTextDataset(train_texts, tokenizer)
print('Tokenizing validation set...')
val_dataset   = FantasyTextDataset(val_texts, tokenizer)
print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)}')

Loading tokenizer from mistralai/Mistral-7B-v0.1...
Vocab size (with special tokens): 32007
Tokenizing training set...
Tokenizing validation set...
Train: 48433 | Val: 5382


## 5. Phase 1 — General Fantasy Fine-Tuning

Load pretrained Mistral-7B, apply a lightweight LoRA (r=16) on the full corpus to teach
it the `[TAG]` format, fantasy vocabulary, and multi-character dialogue structure.
After training, **merge** the adapter into the base weights and save — Phase 2 loads
this merged model as its starting point.

In [9]:
def _load_mistral(model_id_or_path, use_cache=False):
    """Load Mistral in bf16 with FlashAttention 2 when available."""
    kwargs = dict(torch_dtype=torch.bfloat16, device_map='auto')
    try:
        model = AutoModelForCausalLM.from_pretrained(
            model_id_or_path, attn_implementation='flash_attention_2', **kwargs
        )
        print('  Using FlashAttention 2')
    except Exception:
        model = AutoModelForCausalLM.from_pretrained(model_id_or_path, **kwargs)
        print('  Using standard attention')
    model.config.use_cache = use_cache
    return model


print(f'Loading {BASE_MODEL_ID}...')
p1_model = _load_mistral(BASE_MODEL_ID)
p1_model.resize_token_embeddings(len(tokenizer))

p1_lora = LoraConfig(
    r=P1_LORA_R,
    lora_alpha=P1_LORA_ALPHA,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
p1_model = get_peft_model(p1_model, p1_lora)
p1_model.print_trainable_parameters()

p1_args = TrainingArguments(
    output_dir=str(MODEL_DIR / 'phase1_ckpts'),
    num_train_epochs=P1_EPOCHS,
    per_device_train_batch_size=P1_BATCH_SIZE,
    per_device_eval_batch_size=P1_BATCH_SIZE,
    gradient_accumulation_steps=P1_GRAD_ACCUM,
    learning_rate=P1_LR,
    warmup_steps=100,
    weight_decay=0.01,
    bf16=True,
    optim='adamw_torch_fused',
    logging_steps=50,
    save_steps=500,
    eval_strategy='steps',
    eval_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    dataloader_num_workers=8,
    report_to='none',
)

p1_trainer = Trainer(
    model=p1_model,
    args=p1_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

print('\nPhase 1: fine-tuning on full fantasy corpus...')
p1_trainer.train()

print('\nMerging Phase 1 LoRA into base weights...')
p1_model = p1_model.merge_and_unload()
fantasy_base_path = MODEL_DIR / 'fantasy_base_merged'
p1_model.save_pretrained(str(fantasy_base_path))
tokenizer.save_pretrained(str(fantasy_base_path))
print(f'Fantasy base saved to {fantasy_base_path}')

del p1_model
torch.cuda.empty_cache()
print('Phase 1 complete. GPU cache cleared.')

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading mistralai/Mistral-7B-v0.1...


Loading weights: 100%|██████████| 291/291 [00:11<00:00, 26.07it/s]
[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
[transformers] The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


  Using standard attention


[RANK 0] Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


...

Step,Training Loss,Validation Loss
500,1.460950,1.532395
1000,1.435186,1.519191
1500,1.450815,1.509657
2000,1.404977,1.506957
2500,1.394788,1.501916
3000,1.402474,1.498956
3028,1.402474,1.498953


/home/rja3097/.conda/envs/cuda_env/lib/python3.11/site-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/home/rja3097/.conda/envs/cuda_env/lib/python3.11/site-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/home/rja3097/.conda/envs/cuda_env/lib/python3.11/site-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/home/rja3097/.conda/envs/cuda_env/lib/python3.11/site-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/home/rja3097/.conda/envs/cuda_env/lib/python3.11/site-packages/peft/utils/save_and_load


Merging Phase 1 LoRA into base weights...


Writing model shards: 100%|██████████| 1/1 [00:09<00:00,  9.01s/it]

Fantasy base saved to fantasy_v2/models/fantasy_base_merged
Phase 1 complete. GPU cache cleared.


## 6. Phase 2 — Per-Agent LoRA Adapters

Train **5 adapters** on top of `fantasy_base_merged`:
- **narrator** — trained on `[NARRATOR]`-tagged prose from WritingPrompts + Gutenberg
- **wizard / knight / rogue / oracle** — trained on semantically matched corpus examples

Each adapter is saved separately; the base weights are never modified again.
At inference time, one model instance hosts all 5 adapters and hot-swaps between them.

In [16]:
CHARACTERS = {
    # ── Original 4 ──────────────────────────────────────────────────────────
    'wizard':      'An ancient and wise spellcaster who speaks in riddles and arcane knowledge.',
    'knight':      'A honorable warrior sworn to protect the realm, speaks formally and with valor.',
    'rogue':       'A cunning and sarcastic thief who lives by wit and stealth.',
    'oracle':      'A mysterious seer who speaks in cryptic prophecies and half-truths.',
    # ── New 8 ────────────────────────────────────────────────────────────────
    'bard':        'A silver-tongued storyteller and musician who charms listeners with crafted words.',
    'ranger':      'A solitary wilderness tracker who speaks plainly and trusts nature over civilization.',
    'necromancer': 'A cold and scholarly dealer in death magic who views mortality as a resource.',
    'paladin':     'A righteous holy warrior who speaks with divine conviction and unwavering faith.',
    'barbarian':   'A fierce and blunt warrior from the northern wastes who values strength and plain truth.',
    'sea_captain': 'A grizzled nautical commander who barks orders and navigates by instinct and salt air.',
    'druid':       'A patient guardian of ancient forests who speaks through metaphor and natural law.',
    'assassin':    'A cold and precise killer who speaks only when necessary, measuring every word.',
}

NARRATOR_DESCRIPTION = (
    'An atmospheric storyteller who sets vivid scenes, captures mood, '
    'and provides narrative transitions with rich immersive prose.'
)

# Strings that identify Reddit / WritingPrompts meta-content.
# Defined here (before cell-agents) so Phase 2 training also benefits from filtering.
_REDDIT_MARKERS = {
    '/r/', '/u/', 'upvoted', 'writingprompt', 'r/writing',
    '[wp]', '(wp)', 'edit:', '&amp;', 'http://', 'https://',
    'comment', 'subreddit', 'karma', 'moderator',
}

# ── Pre-compute corpus embeddings once (reused for all character adapters) ──
print('Pre-computing semantic embeddings for data selection...')
_char_encoder      = SentenceTransformer('all-MiniLM-L6-v2')
_corpus_embeddings = _char_encoder.encode(
    all_texts, batch_size=256, show_progress_bar=True, convert_to_numpy=True
)
print(f'Embeddings: {_corpus_embeddings.shape}')

# Pre-compute Reddit filter mask once (reused across all character adapters)
_clean_mask = np.array([
    not any(m in t.lower() for m in _REDDIT_MARKERS)
    for t in all_texts
])
_clean_texts = [t for t, keep in zip(all_texts, _clean_mask) if keep]
_clean_embs  = _corpus_embeddings[_clean_mask]
print(f'Clean examples after Reddit filter: {len(_clean_texts)}/{len(all_texts)}')


def build_narrator_examples(base_texts, n_samples=500):
    narrator_texts = [t for t in base_texts if '[NARRATOR]' in t]
    random.shuffle(narrator_texts)
    selected = narrator_texts[:n_samples]
    print(f'  narrator: {len(selected)} examples (pool: {len(narrator_texts)})')
    return selected


def build_character_examples(char_name, description, n_samples=500):
    """Top-n corpus examples most similar to char description, Reddit-filtered."""
    desc_emb = _char_encoder.encode([description], convert_to_numpy=True)
    q      = desc_emb / (np.linalg.norm(desc_emb) + 1e-9)
    c_norm = np.linalg.norm(_clean_embs, axis=1, keepdims=True)
    c      = _clean_embs / (c_norm + 1e-9)
    sims   = (q @ c.T).squeeze()
    top_idx = np.argsort(sims)[::-1][:n_samples]
    matched = [_clean_texts[i] for i in top_idx]
    tag = char_name.upper()
    while len(matched) < min(n_samples, 20):
        matched.append(
            f'[CHARACTER] {char_name.capitalize()}\n'
            f'[PERSONA] {description}\n'
            f'[USER] What brings you here?\n'
            f'[{tag}] I am {char_name.capitalize()}, and I shall play my part.'
        )
    print(f'  {char_name}: {len(matched)} examples  (top-1 sim={sims[top_idx[0]]:.3f})')
    return matched


def train_lora_adapter(char_name, description, base_model_path, base_texts, is_narrator=False):
    print(f'\n{"="*50}')
    print(f'Training adapter: {char_name}')
    print(f'{"="*50}')

    model = _load_mistral(str(base_model_path))

    lora_cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=P2_LORA_R,
        lora_alpha=P2_LORA_ALPHA,
        lora_dropout=P2_LORA_DROPOUT,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
        bias='none',
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()

    examples = (
        build_narrator_examples(base_texts, P2_N_SAMPLES)
        if is_narrator else
        build_character_examples(char_name, description, P2_N_SAMPLES)
    )
    dataset = FantasyTextDataset(examples, tokenizer)

    args = TrainingArguments(
        output_dir=str(LORA_DIR / char_name),
        num_train_epochs=P2_EPOCHS,
        per_device_train_batch_size=P2_BATCH_SIZE,
        gradient_accumulation_steps=P2_GRAD_ACCUM,
        learning_rate=P2_LR,
        bf16=True,
        optim='adamw_torch_fused',
        logging_steps=20,
        save_strategy='no',        # save only at end — avoids mid-run disk spikes
        load_best_model_at_end=False,
        report_to='none',
    )
    Trainer(
        model=model,
        args=args,
        train_dataset=dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    ).train()

    adapter_path = LORA_DIR / char_name
    model.save_pretrained(str(adapter_path))
    print(f'Adapter saved: {adapter_path}')

    del model
    torch.cuda.empty_cache()
    return str(adapter_path)


# Train narrator adapter
train_lora_adapter('narrator', NARRATOR_DESCRIPTION, fantasy_base_path, all_texts, is_narrator=True)

# Train all character adapters
for char_name, char_desc in CHARACTERS.items():
    train_lora_adapter(char_name, char_desc, fantasy_base_path, all_texts)

print(f'\nAll {len(CHARACTERS) + 1} adapters trained.')


Pre-computing semantic embeddings for data selection...


Batches: 100%|██████████| 427/427 [00:27<00:00, 15.70it/s] 


Embeddings: (109127, 384)
Clean examples after Reddit filter: 107985/109127

Training adapter: narrator


Loading weights: 100%|██████████| 291/291 [00:04<00:00, 62.67it/s]


  Using standard attention
trainable params: 54,525,952 || all params: 7,296,315,392 || trainable%: 0.7473
  narrator: 500 examples (pool: 22566)


...

Loading weights: 100%|██████████| 291/291 [00:04<00:00, 63.53it/s]


  Using standard attention
trainable params: 54,525,952 || all params: 7,296,315,392 || trainable%: 0.7473
  assassin: 500 examples  (top-1 sim=0.450)


[RANK 0] Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Step,Training Loss
20,2.170122
40,2.019501
60,1.863675
80,1.717968


Adapter saved: fantasy_v2/lora_adapters/assassin

All 13 adapters trained.


## 7. RAG Setup — Lore Vector Store

In [3]:
LORE_DOCUMENTS = [
    'The Dragon Wars began in the Third Age when the great wyrm Malachar rose from the Ashen Mountains, scorching three kingdoms before being bound by the Conclave.',
    'Wizards of the Silver Tower are bound by an oath never to use fire magic within city walls.',
    'The Knights of the Crimson Order were established to protect the kingdom from demon incursions, each sworn to die before retreat.',
    "Rogues' guilds operate in the shadow markets beneath every major city, trading in secrets as much as stolen goods.",
    'Bards carry the Seal of Free Passage, granting them entry to any tavern or noble court in exchange for songs and news.',
    'The ancient elven script, Quethara, holds the secrets of the first enchantments — written in solidified starlight.',
    'The Moonstone Prophecy foretells a hero rising from the common folk to unite the fractured kingdoms against the coming dark.',
    "Healing potions require dragon's breath moss, only found near volcanic vents in the eastern ranges, harvested at the new moon.",
    'The Underdark is a vast subterranean world populated by creatures that shun daylight and follow their own cruel laws.',
    'Magic is drawn from ley lines that criss-cross the land, converging at ancient standing stones erected by the First People.',
    'The Veil Between Worlds grows thin on the night of the Blood Moon, allowing spirits to pass freely between the living and dead realms.',
    'Sea witches trade in bottled storms and drowned memories; their price is always something you cannot name until it is gone.',
    'The Whispering Wood is forbidden to travelers — those who enter hear voices that slowly replace their own thoughts.',
    "An oracle's prophecy is never wrong, but it is never complete — it always describes the worst possible interpretation of the truth.",
    "The merchant guilds of the Free Cities control every trade route; no caravan moves without their blessing.",
]


class LoreRAG:
    def __init__(self, documents, model_name='all-MiniLM-L6-v2'):
        self.documents = documents
        self.embedder  = SentenceTransformer(model_name)
        embeddings = self.embedder.encode(documents, convert_to_numpy=True)
        self.index = faiss.IndexFlatL2(embeddings.shape[1])
        self.index.add(embeddings.astype(np.float32))
        print(f'Lore index ready — {self.index.ntotal} entries')

    def retrieve(self, query, top_k=3):
        q_emb = self.embedder.encode([query], convert_to_numpy=True).astype(np.float32)
        _, idx = self.index.search(q_emb, top_k)
        return [self.documents[i] for i in idx[0] if i < len(self.documents)]

    def save(self, path):
        path = Path(path)
        faiss.write_index(self.index, str(path / 'lore.index'))
        with open(path / 'documents.json', 'w') as f:
            json.dump(self.documents, f)

    @classmethod
    def load(cls, path, model_name='all-MiniLM-L6-v2'):
        path = Path(path)
        with open(path / 'documents.json') as f:
            docs = json.load(f)
        obj = cls.__new__(cls)
        obj.documents = docs
        obj.embedder  = SentenceTransformer(model_name)
        obj.index     = faiss.read_index(str(path / 'lore.index'))
        return obj


rag = LoreRAG(LORE_DOCUMENTS)
rag.save(RAG_DIR)
print('RAG saved.')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 28365.94it/s]


Lore index ready — 15 entries
RAG saved.


## 8. Agent System

**Memory layout**: one Mistral-7B instance (~15 GB bf16) with all 5 LoRA adapters loaded
simultaneously. `set_adapter()` switches the active adapter in microseconds — no model
reloads, no extra VRAM per agent.

```
SharedFantasyModel
  └── Mistral-7B weights (frozen, bf16)
       ├── adapter: narrator
       ├── adapter: wizard
       ├── adapter: knight
       ├── adapter: rogue
       └── adapter: oracle   (+ any dynamically minted ones)
```

In [4]:
# ── Shared Model ──────────────────────────────────────────────────────────

class SharedFantasyModel:
    def __init__(self, model_path, tokenizer, lora_dir, adapter_names):
        self.tokenizer = tokenizer
        self._loaded   = []

        print('Loading fantasy base model...')
        base = _load_mistral(model_path, use_cache=True)

        self.model = None
        for name in adapter_names:
            path = Path(lora_dir) / name
            if not path.exists():
                print(f'  WARNING: no adapter at {path}')
                continue
            if self.model is None:
                self.model = PeftModel.from_pretrained(base, str(path), adapter_name=name)
                print(f'  Initialised with adapter: {name}')
            else:
                self.model.load_adapter(str(path), adapter_name=name)
                print(f'  Loaded adapter: {name}')
            self._loaded.append(name)

        if self.model is None:
            print('No adapters found — using bare fantasy_base_merged model.')
            self.model = base

        self.model.eval()
        print(f'SharedFantasyModel ready. Adapters: {self._loaded}')

    def set_active(self, name):
        # name=None means "use whatever adapter is already active" (bare-model mode)
        if name is not None and name in self._loaded:
            self.model.set_adapter(name)

    def add_adapter(self, name, adapter_path):
        self.model.load_adapter(str(adapter_path), adapter_name=name)
        self._loaded.append(name)
        print(f'Dynamically loaded adapter: {name}')

    @torch.no_grad()
    def generate(self, prompt, max_new_tokens=120, temperature=0.75,
                 top_p=0.90, repetition_penalty=1.5):
        inputs = self.tokenizer(
            prompt,
            return_tensors='pt',
            truncation=True,
            max_length=MAX_SEQ_LEN - max_new_tokens,
        ).to(self.model.device)

        output = self.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            pad_token_id=self.tokenizer.eos_token_id,
        )
        generated = output[0][inputs['input_ids'].shape[1]:]
        return self.tokenizer.decode(generated, skip_special_tokens=True)


# ── Response cleaner ──────────────────────────────────────────────────────

_HARD_STOPS = [
    '[USER]', '[SETTING]', '[CHARACTER]', '[NARRATOR]',
    '[LORE]', '[STORY SO FAR]', '[PERSONA]',
    '<|eot|>', '<|eot', '<|endoftext|>', '<|im_end|>',
]

# Newline-anchored artifact patterns (stage directions, metadata headers)
_ARTIFACT_RE = re.compile(
    r'\n\s*_{2,}'
    r'|\n\s*\*[^\n]*'
    r'|\n\s*SIONS:'
    r'|\n\s*[Cc]haracter\s+[Nn]ame'
    r'|\n\s*ACTION\s+'
    r'|\n\s*EVENT\s+'
    r'|\n\s*CHATTER'
    r'|\n\s*CROWD\s+'
    r'|\n\s*END\s+SCENE'
    r'|\n\s*END\s+OF',
)

# Inline markers that indicate the model is generating Reddit / WritingPrompts meta-content.
# Cut back to the nearest sentence boundary when any of these appear.
_INLINE_STOPS = re.compile(
    r'/r/|'
    r'/u/|'
    r'upvoted|'
    r'WritingPrompt|'
    r'writingprompt|'
    r'\[wp\]|'
    r'\(wp\)|'
    r'r/writing|'
    r'ReplyGive Award|'
    r'showParentComments|'
    r'window\.location|'
    r'DMG page|'
    r'4th Edition|'
    r'Congratulations.*day|'
    r'short story prompt was provided',
    re.IGNORECASE
)

# Strip inline LIGHT item tags like [SWORD], [/RING], [FLINT+STEEL]
_LIGHT_TAG_RE = re.compile(r'\[/?[A-Za-z][A-Za-z0-9_+/\s]*\]')


def _clean_response(text, max_sentences=3):
    # 1. Cut at first hard-stop token
    for stop in _HARD_STOPS:
        idx = text.find(stop)
        if idx != -1:
            text = text[:idx]

    # 2. Cut at next format tag on a new line
    m = re.search(r'\n\[', text)
    if m:
        text = text[:m.start()]

    # 3. Cut at newline-anchored artifact patterns
    m = _ARTIFACT_RE.search(text)
    if m:
        text = text[:m.start()]

    # 4. Cut at inline Reddit / WritingPrompts bleed-through.
    #    Back up to the nearest sentence boundary so we don't chop mid-word.
    m = _INLINE_STOPS.search(text)
    if m:
        boundary = max(
            text.rfind('.', 0, m.start()),
            text.rfind('!', 0, m.start()),
            text.rfind('?', 0, m.start()),
        )
        text = text[:boundary + 1] if boundary >= 0 else text[:m.start()]

    # 5. Strip inline LIGHT item tags (use space so joined words don't merge)
    text = _LIGHT_TAG_RE.sub(' ', text)

    # 6. Collapse excess whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # 7. Sentence truncation — hard cap on length
    sentences = re.split(r'(?<=[.!?])\s+', text)
    text = ' '.join(s.strip() for s in sentences[:max_sentences] if s.strip())

    return text.strip()


# ── Narrator Agent ────────────────────────────────────────────────────────

class NarratorAgent:
    ADAPTER = 'narrator'

    def __init__(self, shared, rag):
        self.shared = shared
        self.rag    = rag

    def _prompt(self, user_input, story_context):
        lore = self.rag.retrieve(user_input + ' ' + story_context[:200])
        lore_block = '\n'.join(f'[LORE] {l}' for l in lore[:2])
        return (
            f'{lore_block}\n'
            f'[SETTING] {user_input}\n'
            f'[STORY SO FAR] {story_context[-400:]}\n'
            f'[NARRATOR]'
        )

    def _transition_prompt(self, story_context):
        return f'[STORY SO FAR] {story_context[-400:]}\n[NARRATOR]'

    def open_scene(self, user_input, story_context=''):
        self.shared.set_active(self.ADAPTER)
        raw = self.shared.generate(
            self._prompt(user_input, story_context), max_new_tokens=100, temperature=0.85)
        return _clean_response(raw, max_sentences=4)

    def close_scene(self, story_context):
        self.shared.set_active(self.ADAPTER)
        raw = self.shared.generate(
            self._transition_prompt(story_context), max_new_tokens=70, temperature=0.80)
        return _clean_response(raw, max_sentences=3)


# ── Character Agent ───────────────────────────────────────────────────────

class CharacterAgent:
    def __init__(self, name, description, shared, rag, adapter_name=None):
        self.name         = name
        self.desc         = description
        self.shared       = shared
        self.rag          = rag
        self.adapter_name = adapter_name  # None = bare base-model mode

    def respond(self, story_context, latest_line=''):
        # Only switch adapters when one is assigned; bare-model mode skips this
        if self.adapter_name is not None:
            self.shared.set_active(self.adapter_name)
        query = latest_line or story_context[:200]
        lore  = self.rag.retrieve(query)
        lore_block = '\n'.join(f'[LORE] {l}' for l in lore[:2])
        tag = self.name.upper()
        recent_ctx = story_context[-500:]
        prompt = (
            f'[CHARACTER] {self.name}\n'
            f'[PERSONA] {self.desc}\n'
            f'{lore_block}\n'
            f'[STORY SO FAR] {recent_ctx}\n'
            f'[{tag}]'
        )
        raw = self.shared.generate(prompt, max_new_tokens=100)
        return _clean_response(raw, max_sentences=3)


# ── Character Factory ─────────────────────────────────────────────────────

CANONICAL_ADAPTERS = [
    'narrator',
    'wizard', 'knight', 'rogue', 'oracle',
    'bard', 'ranger', 'necromancer', 'paladin',
    'barbarian', 'sea_captain', 'druid', 'assassin',
]

# Strings that identify Reddit/WritingPrompts meta-content — used to filter
# training examples so adapters don't memorise Reddit formatting.
_REDDIT_MARKERS = {
    '/r/', '/u/', 'upvoted', 'writingprompt', 'r/writing',
    '[wp]', '(wp)', 'edit:', '&amp;', 'http://', 'https://',
    'comment', 'subreddit', 'karma', 'moderator',
}


def add_character_tokens(tokenizer, character_keys):
    new_tags = [f'[{k.upper()}]' for k in character_keys]
    existing = set(tokenizer.all_special_tokens)
    to_add   = [t for t in new_tags if t not in existing]
    if to_add:
        tokenizer.add_special_tokens({'additional_special_tokens': to_add})


class CharacterFactory:
    _SYSTEM = (
        'You are a creative writing director for a fantasy story. '
        f'Given a premise, create EXACTLY {NUM_CHARACTERS} characters. '
        'Respond ONLY with a valid JSON object containing exactly '
        f'{NUM_CHARACTERS} keys. Each key is a snake_case character name, '
        'each value is a one-sentence persona description.\n\n'
        'Example (ghost ship story):\n'
        '{"cursed_captain": "A hollow-eyed captain bound by an ancient oath.", '
        '"spectral_navigator": "A ghost who charts courses between worlds.", '
        '"desperate_survivor": "The last living crew member, half-mad from isolation.", '
        '"sea_witch": "A conjurer who trades in lost souls and drowned secrets."}\n\n'
        f'Output ONLY the JSON with exactly {NUM_CHARACTERS} entries. No commentary.'
    )
    _DEFAULTS = [
        ('wandering_hero',    'A brave traveler who trusts deeds over words.'),
        ('ancient_elder',     'A figure who speaks in warnings and half-remembered prophecies.'),
        ('cunning_trickster', 'A shape-shifter who manipulates events from the shadows.'),
        ('fallen_guardian',   'A once-noble protector consumed by guilt, seeking redemption.'),
    ]

    def __init__(self):
        print('[CharacterFactory] Loading TinyLlama director...')
        self._director = pipeline(
            'text-generation',
            model='TinyLlama/TinyLlama-1.1B-Chat-v1.0',
            device=0 if torch.cuda.is_available() else -1,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        )
        print('[CharacterFactory] Ready.')

    def generate(self, premise):
        prompt = (
            f'<|system|>\n{self._SYSTEM}\n'
            f'<|user|>\nStory premise: {premise}\n'
            f'<|assistant|>\n{{'
        )
        out = self._director(
            prompt, max_new_tokens=200, do_sample=False, repetition_penalty=1.1
        )[0]['generated_text']
        raw   = '{' + out[len(prompt):]
        chars = self._parse(raw)
        return self._enforce_count(chars)

    def _parse(self, raw):
        match = re.search(r'\{.*\}', raw, re.DOTALL)
        if not match:
            print('[CharacterFactory] Parse failed, using defaults.')
            return {}
        try:
            d = json.loads(match.group())
            result = {}
            for k, v in d.items():
                if isinstance(v, dict):
                    desc = v.get('description') or v.get('persona') or v.get('name') or str(v)
                else:
                    desc = str(v)
                result[k.lower().replace(' ', '_')] = desc
            return result
        except json.JSONDecodeError:
            return {}

    def _enforce_count(self, chars):
        for name, desc in self._DEFAULTS:
            if len(chars) >= NUM_CHARACTERS:
                break
            if name not in chars:
                chars[name] = desc
        return dict(list(chars.items())[:NUM_CHARACTERS])


# ── Dataset + helpers for dynamic adapter training ────────────────────────

class FantasyTextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=MAX_SEQ_LEN):
        self.examples = []
        for text in texts:
            enc = tokenizer(
                text, truncation=True, max_length=max_length,
                padding='max_length', return_tensors='pt',
            )
            self.examples.append({
                'input_ids':      enc['input_ids'].squeeze(),
                'attention_mask': enc['attention_mask'].squeeze(),
                'labels':         enc['input_ids'].squeeze().clone(),
            })
    def __len__(self):          return len(self.examples)
    def __getitem__(self, idx): return self.examples[idx]


def build_narrator_examples(base_texts, n_samples=500):
    narrator_texts = [t for t in base_texts if '[NARRATOR]' in t]
    random.shuffle(narrator_texts)
    selected = narrator_texts[:n_samples]
    print(f'  narrator: {len(selected)} examples (pool: {len(narrator_texts)})')
    return selected


def build_character_examples(char_name, description, base_texts, n_samples=500):
    # Exclude WritingPrompts examples — they contain Reddit meta-content that
    # causes the adapter to memorise Reddit formatting instead of dialogue style.
    clean_texts = [
        t for t in base_texts
        if not any(m in t.lower() for m in _REDDIT_MARKERS)
    ]
    print(f'  {char_name}: using {len(clean_texts)}/{len(base_texts)} clean examples')

    desc_emb = _char_encoder.encode([description], convert_to_numpy=True)
    clean_embs = _char_encoder.encode(clean_texts, batch_size=256, convert_to_numpy=True)
    q      = desc_emb / (np.linalg.norm(desc_emb) + 1e-9)
    c_norm = np.linalg.norm(clean_embs, axis=1, keepdims=True)
    c      = clean_embs / (c_norm + 1e-9)
    sims   = (q @ c.T).squeeze()
    top_idx = np.argsort(sims)[::-1][:n_samples]
    matched = [clean_texts[i] for i in top_idx]
    tag = char_name.upper()
    while len(matched) < min(n_samples, 20):
        matched.append(
            f'[CHARACTER] {char_name.capitalize()}\n'
            f'[PERSONA] {description}\n'
            f'[USER] What brings you here?\n'
            f'[{tag}] I am {char_name.capitalize()}, and I shall play my part.'
        )
    print(f'  {char_name}: {len(matched)} examples  (top-1 sim={sims[top_idx[0]]:.3f})')
    return matched


def train_lora_adapter(char_name, description, base_model_path, base_texts, is_narrator=False):
    print(f'\n{"="*50}\nTraining adapter: {char_name}\n{"="*50}')
    model = _load_mistral(str(base_model_path))
    lora_cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=P2_LORA_R, lora_alpha=P2_LORA_ALPHA, lora_dropout=P2_LORA_DROPOUT,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'], bias='none',
    )
    model = get_peft_model(model, lora_cfg)
    examples = (
        build_narrator_examples(base_texts, P2_N_SAMPLES) if is_narrator
        else build_character_examples(char_name, description, base_texts, P2_N_SAMPLES)
    )
    dataset = FantasyTextDataset(examples, tokenizer)
    args = TrainingArguments(
        output_dir=str(LORA_DIR / char_name),
        num_train_epochs=P2_EPOCHS,
        per_device_train_batch_size=P2_BATCH_SIZE,
        gradient_accumulation_steps=P2_GRAD_ACCUM,
        learning_rate=P2_LR,
        bf16=True,
        optim='adamw_torch_fused',
        logging_steps=20,
        save_strategy='no',
        load_best_model_at_end=False,
        report_to='none',
    )
    Trainer(
        model=model, args=args, train_dataset=dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    ).train()
    model.save_pretrained(str(LORA_DIR / char_name))
    print(f'Adapter saved: {LORA_DIR / char_name}')
    del model
    torch.cuda.empty_cache()


def _closest_adapter(description, loaded_adapters, char_encoder):
    """Map to the most semantically similar canonical pre-trained adapter."""
    pool = [a for a in loaded_adapters if a in CANONICAL_ADAPTERS] or loaded_adapters
    desc_emb   = char_encoder.encode([description], convert_to_numpy=True)
    adapt_embs = char_encoder.encode(pool, convert_to_numpy=True)
    sims = (desc_emb / np.linalg.norm(desc_emb)) @ (adapt_embs / np.linalg.norm(adapt_embs, axis=1, keepdims=True)).T
    return pool[int(np.argmax(sims))]


# ── Story Orchestrator ────────────────────────────────────────────────────

class StoryOrchestrator:
    CONTEXT_WINDOW = 12

    def __init__(self, shared, tokenizer, rag, lora_dir, all_texts, fantasy_base_path):
        self.shared            = shared
        self.tokenizer         = tokenizer
        self.rag               = rag
        self.lora_dir          = Path(lora_dir)
        self.all_texts         = all_texts
        self.fantasy_base_path = fantasy_base_path

        self.narrator  = NarratorAgent(shared, rag)
        self.factory   = CharacterFactory()
        self.agents    = []
        self.history   = []
        self._ready    = False

    def _initialise(self, premise, characters=None):
        if characters is not None:
            print(f'[Orchestrator] Using provided cast: {list(characters.keys())}')
        else:
            print('[Orchestrator] Generating characters from premise...')
            characters = self.factory.generate(premise)
            print(f'[Orchestrator] Generated cast: {list(characters.keys())}')

        add_character_tokens(self.tokenizer, list(characters.keys()))
        self.shared.model.resize_token_embeddings(len(self.tokenizer))

        canonical_available = [a for a in self.shared._loaded if a in CANONICAL_ADAPTERS]
        if not canonical_available:
            print('[Orchestrator] No canonical adapters on disk — running all characters '
                  'on the base fantasy_base_merged model (no LoRA). '
                  'Re-run Phase 2 training to restore per-character adapters.')

        for name, desc in characters.items():
            adapter_path = self.lora_dir / name
            if adapter_path.exists() and name in CANONICAL_ADAPTERS:
                # Pre-trained canonical adapter on disk — load and use directly
                if name not in self.shared._loaded:
                    self.shared.add_adapter(name, str(adapter_path))
                    canonical_available.append(name)
                adapter_name = name
            elif canonical_available:
                # Canonical adapters exist — semantic fallback, no training needed
                adapter_name = _closest_adapter(desc, self.shared._loaded, _char_encoder)
                print(f'[Orchestrator] "{name}" → adapter "{adapter_name}" (semantic fallback)')
            else:
                # Nothing on disk at all — use bare merged model rather than training
                # on WritingPrompts-contaminated data.
                adapter_name = None
                print(f'[Orchestrator] "{name}" → bare base model (no adapter)')

            self.agents.append(CharacterAgent(name, desc, self.shared, self.rag,
                                              adapter_name=adapter_name))
            print(f'[Orchestrator] Agent ready: {name} (adapter: {adapter_name})')

        self._ready = True

    @property
    def story_context(self):
        recent = self.history[-self.CONTEXT_WINDOW:]
        return ' '.join(f'[{spk}] {txt}' for spk, txt in recent)

    def beat(self, user_input, characters=None):
        if not self._ready:
            self._initialise(user_input, characters=characters)

        self.history.append(('USER', user_input))
        result = {}

        n_open = self.narrator.open_scene(user_input, self.story_context)
        self.history.append(('NARRATOR', n_open))
        result['narrator_open'] = n_open

        result['characters'] = {}
        latest = n_open
        for agent in self.agents:
            line = agent.respond(self.story_context, latest)
            self.history.append((agent.name.upper(), line))
            result['characters'][agent.name] = line
            latest = line

        n_close = self.narrator.close_scene(self.story_context)
        self.history.append(('NARRATOR', n_close))
        result['narrator_close'] = n_close

        return result

    def print_beat(self, result):
        print(f'\n[NARRATOR] {result["narrator_open"]}\n')
        for name, line in result['characters'].items():
            print(f'[{name.upper()}] {line}')
        print(f'\n[NARRATOR] {result["narrator_close"]}\n')
        print('─' * 70)

    def print_full_story(self):
        print('\n' + '=' * 70)
        print('FULL STORY')
        print('=' * 70)
        for speaker, text in self.history:
            print(f'[{speaker}]\n  {text}\n')
        print('=' * 70)


print('All agent classes defined.')


All agent classes defined.


## 9. Inference Demo

Load the shared model with all 5 adapters, create the orchestrator, and run a story.
Characters are generated from the premise — they are not pre-fixed.

In [5]:
# ── CHECKPOINT CELL — Run this after a kernel restart to skip all training ──
# Prerequisites (must run first): cell-imports, cell-config, cell-rag, cell-agents
# All training artifacts must already be saved on disk.

# 1. Re-define _load_mistral (normally lives in cell-phase1)
def _load_mistral(model_id_or_path, use_cache=False):
    kwargs = dict(torch_dtype=torch.bfloat16, device_map='auto')
    try:
        model = AutoModelForCausalLM.from_pretrained(
            model_id_or_path, attn_implementation='flash_attention_2', **kwargs)
        print('  Using FlashAttention 2')
    except Exception:
        model = AutoModelForCausalLM.from_pretrained(model_id_or_path, **kwargs)
        print('  Using standard attention')
    model.config.use_cache = use_cache
    return model

# 2. Verify saved artifacts exist
fantasy_base_path = MODEL_DIR / 'fantasy_base_merged'
assert fantasy_base_path.exists(), f'Missing: {fantasy_base_path} — run Phase 1 training first'
assert (RAG_DIR / 'lore.index').exists(), f'Missing RAG index — run cell-rag first'
assert (DATA_DIR / 'combined_corpus.txt').exists(), f'Missing corpus — run cell-combine first'
print('All artifacts found.')

# 3. Reload corpus (needed by StoryOrchestrator for dynamic adapter training)
with open(DATA_DIR / 'combined_corpus.txt', encoding='utf-8') as f:
    all_texts = f.read().split('\n\n')
all_texts = [t.strip() for t in all_texts if t.strip()]
print(f'Corpus reloaded: {len(all_texts)} examples')

# 4. Recompute corpus embeddings (needed for semantic character data selection)
print('Re-encoding corpus for semantic search...')
_char_encoder = SentenceTransformer('all-MiniLM-L6-v2')
_corpus_embeddings = _char_encoder.encode(
    all_texts, batch_size=256, show_progress_bar=True, convert_to_numpy=True)
print(f'Embeddings: {_corpus_embeddings.shape}')

# 5. Load tokenizer from saved base
tokenizer = AutoTokenizer.from_pretrained(str(fantasy_base_path))
tokenizer.pad_token = tokenizer.eos_token
print(f'Tokenizer loaded. Vocab size: {len(tokenizer)}')

# 6. Load RAG
rag = LoreRAG.load(RAG_DIR)

# 7. Detect available adapters and load shared model
_all_known = [
    'narrator',
    'wizard', 'knight', 'rogue', 'oracle',
    'bard', 'ranger', 'necromancer', 'paladin',
    'barbarian', 'sea_captain', 'druid', 'assassin',
]
available_adapters = [name for name in _all_known if (LORA_DIR / name).exists()]
print(f'Available adapters: {available_adapters}')

shared = SharedFantasyModel(
    model_path=str(fantasy_base_path),
    tokenizer=tokenizer,
    lora_dir=LORA_DIR,
    adapter_names=available_adapters,
)

# 8. Create orchestrator
orchestrator = StoryOrchestrator(
    shared=shared,
    tokenizer=tokenizer,
    rag=rag,
    lora_dir=LORA_DIR,
    all_texts=all_texts,
    fantasy_base_path=str(fantasy_base_path),
)

print('\nSystem ready. Run: orchestrator.beat("<your premise>")')


All artifacts found.
Corpus reloaded: 109127 examples
Re-encoding corpus for semantic search...


Batches: 100%|██████████| 427/427 [01:17<00:00,  5.51it/s]


Embeddings: (109127, 384)
Tokenizer loaded. Vocab size: 32007


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10056.41it/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Available adapters: ['narrator', 'wizard', 'knight', 'rogue', 'oracle', 'bard', 'ranger', 'necromancer', 'paladin', 'barbarian', 'sea_captain', 'druid', 'assassin']
Loading fantasy base model...


...

  Using standard attention
  Initialised with adapter: narrator
  Loaded adapter: wizard
  Loaded adapter: knight
  Loaded adapter: rogue
  Loaded adapter: oracle
  Loaded adapter: bard
  Loaded adapter: ranger
  Loaded adapter: necromancer
  Loaded adapter: paladin
  Loaded adapter: barbarian
  Loaded adapter: sea_captain
  Loaded adapter: druid


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


  Loaded adapter: assassin
SharedFantasyModel ready. Adapters: ['narrator', 'wizard', 'knight', 'rogue', 'oracle', 'bard', 'ranger', 'necromancer', 'paladin', 'barbarian', 'sea_captain', 'druid', 'assassin']
[CharacterFactory] Loading TinyLlama director...


Loading weights: 100%|██████████| 201/201 [00:03<00:00, 56.14it/s]


[CharacterFactory] Ready.

System ready. Run: orchestrator.beat("<your premise>")


In [6]:
# ── INFERENCE SETUP — run this after cell-agents to load the shared model ──
# Cell execution order: cell-agents → THIS cell → cell-demo
# (If you get a NameError on _load_mistral, run the checkpoint cell first.)

fantasy_base_path = MODEL_DIR / 'fantasy_base_merged'

# Load tokenizer from saved base
tokenizer = AutoTokenizer.from_pretrained(str(fantasy_base_path))
tokenizer.pad_token = tokenizer.eos_token
print(f'Tokenizer loaded. Vocab size: {len(tokenizer)}')

# Load RAG
rag = LoreRAG.load(RAG_DIR)

# Load shared model — try all canonical adapters, skip any not yet on disk
available_adapters = [
    name for name in CANONICAL_ADAPTERS
    if (LORA_DIR / name).exists()
]
print(f'Adapters found on disk: {available_adapters}')

shared = SharedFantasyModel(
    model_path=str(fantasy_base_path),
    tokenizer=tokenizer,
    lora_dir=LORA_DIR,
    adapter_names=available_adapters,
)

# Create orchestrator
orchestrator = StoryOrchestrator(
    shared=shared,
    tokenizer=tokenizer,
    rag=rag,
    lora_dir=LORA_DIR,
    all_texts=all_texts,
    fantasy_base_path=str(fantasy_base_path),
)

print('\nSystem ready.')
print('Run cell-demo to start the story.')
print('Use orchestrator.beat(premise) or orchestrator.beat(premise, characters=my_cast)')


Tokenizer loaded. Vocab size: 32007


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 27211.72it/s]


Adapters found on disk: ['narrator', 'wizard', 'knight', 'rogue', 'oracle', 'bard', 'ranger', 'necromancer', 'paladin', 'barbarian', 'sea_captain', 'druid', 'assassin']
Loading fantasy base model...


Loading weights: 100%|██████████| 291/291 [00:15<00:00, 18.36it/s]


  Using standard attention
  Initialised with adapter: narrator
  Loaded adapter: wizard
  Loaded adapter: knight
  Loaded adapter: rogue
  Loaded adapter: oracle
  Loaded adapter: bard
  Loaded adapter: ranger
  Loaded adapter: necromancer
  Loaded adapter: paladin
  Loaded adapter: barbarian
  Loaded adapter: sea_captain
  Loaded adapter: druid
  Loaded adapter: assassin
SharedFantasyModel ready. Adapters: ['narrator', 'wizard', 'knight', 'rogue', 'oracle', 'bard', 'ranger', 'necromancer', 'paladin', 'barbarian', 'sea_captain', 'druid', 'assassin']
[CharacterFactory] Loading TinyLlama director...


Loading weights: 100%|██████████| 201/201 [00:02<00:00, 69.17it/s]


[CharacterFactory] Ready.

System ready.
Run cell-demo to start the story.
Use orchestrator.beat(premise) or orchestrator.beat(premise, characters=my_cast)


In [7]:
# ══════════════════════════════════════════════════════════════════════════
# STORY DEMO — two modes
#
# MODE A (auto):  TinyLlama generates a cast from your premise.
# MODE B (manual): you pick characters from the list below.
#
# Available canonical adapters (each has a dedicated trained LoRA):
# ══════════════════════════════════════════════════════════════════════════

AVAILABLE_ADAPTERS = {
    'wizard':      'Ancient spellcaster — speaks in riddles and arcane knowledge.',
    'knight':      'Honorable warrior — formal, valor-driven, sworn to protect.',
    'rogue':       'Cunning thief — sarcastic, lives by wit and stealth.',
    'oracle':      'Mysterious seer — cryptic prophecies and half-truths.',
    'bard':        'Silver-tongued storyteller — charms with crafted words.',
    'ranger':      'Solitary wilderness tracker — plain-spoken, trusts nature.',
    'necromancer': 'Death-magic scholar — cold, views mortality as a resource.',
    'paladin':     'Holy warrior — divine conviction and unwavering faith.',
    'barbarian':   'Northern wastes warrior — blunt, values strength and truth.',
    'sea_captain': 'Nautical commander — barks orders, navigates by instinct.',
    'druid':       'Forest guardian — patient, speaks in metaphor and natural law.',
    'assassin':    'Precise killer — measures every word, speaks only when needed.',
}

print('─' * 62)
print('AVAILABLE CHARACTER ADAPTERS')
print('─' * 62)
for name, desc in AVAILABLE_ADAPTERS.items():
    print(f'  {name:<14}  {desc}')
print('─' * 62)


# ──────────────────────────────────────────────────────────────────────────
# ► CONFIGURE YOUR STORY HERE — edit these two sections
# ──────────────────────────────────────────────────────────────────────────

# Any story context — no restrictions on setting or genre
beats = [
    'We stand at the entrance to the Ashen Catacombs beneath the ruined city of Valdren. '
    'The air smells of old fire and something older. Strange sigils glow faintly on the walls.',

    'A sealed iron door blocks the passage. Carved into it: a warning in the First Tongue '
    'that none of us can fully read.',

    'Beyond the door we find a chamber filled with crystallised time — '
    'frozen moments suspended in amber light.',
]

# ── MODE A: set to True to let TinyLlama auto-generate cast from premise ──
use_auto_cast = False

# ── MODE B: choose your own cast ─────────────────────────────────────────
# • Use any key from AVAILABLE_ADAPTERS above to get that exact trained adapter.
# • Use any other name and it semantic-fallbacks to the closest canonical adapter.
# • Descriptions are yours to write — they shape each character's RAG context.
# • Include as many or as few characters as you like (2–6 works well).
my_cast = {
    'wizard':      'An ancient and weary spellcaster who has watched empires crumble.',
    'rogue':       'A quick-fingered thief here purely for whatever she can steal.',
    'necromancer': 'A pale scholar who hears the dead whispering in every shadow.',
    'oracle':      'A blind seer who perceives only futures, never the present moment.',
}

# ──────────────────────────────────────────────────────────────────────────

characters_arg = None if use_auto_cast else my_cast

for premise in beats:
    print(f'\n[USER] {premise}')
    result = orchestrator.beat(premise, characters=characters_arg)
    orchestrator.print_beat(result)

orchestrator.print_full_story()


──────────────────────────────────────────────────────────────

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`



AVAILABLE CHARACTER ADAPTERS
──────────────────────────────────────────────────────────────
  wizard          Ancient spellcaster — speaks in riddles and arcane knowledge.
  knight          Honorable warrior — formal, valor-driven, sworn to protect.
  rogue           Cunning thief — sarcastic, lives by wit and stealth.
  oracle          Mysterious seer — cryptic prophecies and half-truths.
  bard            Silver-tongued storyteller — charms with crafted words.
  ranger          Solitary wilderness tracker — plain-spoken, trusts nature.
  necromancer     Death-magic scholar — cold, views mortality as a resource.
  paladin         Holy warrior — divine conviction and unwavering faith.
  barbarian       Northern wastes warrior — blunt, values strength and truth.
  sea_captain     Nautical commander — barks orders, navigates by instinct.
  druid           Forest guardian — patient, speaks in metaphor and natural law.
  assassin        Precise killer — measures every word, speaks only wh

[transformers] The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


[Orchestrator] Agent ready: wizard (adapter: wizard)
[Orchestrator] Agent ready: rogue (adapter: rogue)
[Orchestrator] Agent ready: necromancer (adapter: necromancer)
[Orchestrator] Agent ready: oracle (adapter: oracle)

[NARRATOR] A massive stone door stands before us, carved with images from long-forgotten legends - half obscured now in shadow after millennia spent undisturbed underground.''It lies beyond this portal,' I tell my companions,'the key we have sought for so many years...'I draw Steelthorn as she stirs in her sleep beside me''This will be our greatest adventure yet! Let no man say of you or your sword what they do not dare speak themselves...''You

[WIZARD] an old scroll case made out leather with metal reinforcement; inside are several rolled up parchments of various ages ``A map? To where?'' I ask excitedly, pointing at it. ``To a hidden temple deep within the mountains," he replies cryptically.
[ROGUE] An old dwarvish proverb: All that glitters is gold; all that shines

## 10. Gradio UI — HPC Deployment

Run the cell below to launch a Gradio interface directly on the HPC.
`share=True` prints a public `gradio.live` URL valid for 72 hours.

**Prerequisites:** `cell-agents` and `cell-setup` (or checkpoint cell) must have been run
so that `shared`, `tokenizer`, `_char_encoder`, `LORA_DIR`, `all_texts`, and
`fantasy_base_path` are all in memory.


In [12]:
import gradio as gr

# ── UI constants ───────────────────────────────────────────────────────────

_AVAILABLE_ADAPTERS = {
    'wizard':      'Ancient spellcaster — speaks in riddles and arcane knowledge.',
    'knight':      'Honorable warrior — formal, valor-driven, sworn to protect.',
    'rogue':       'Cunning thief — sarcastic, lives by wit and stealth.',
    'oracle':      'Mysterious seer — cryptic prophecies and half-truths.',
    'bard':        'Silver-tongued storyteller — charms with crafted words.',
    'ranger':      'Solitary wilderness tracker — plain-spoken, trusts nature.',
    'necromancer': 'Death-magic scholar — cold, views mortality as a resource.',
    'paladin':     'Holy warrior — divine conviction and unwavering faith.',
    'barbarian':   'Northern wastes warrior — blunt, values strength and truth.',
    'sea_captain': 'Nautical commander — barks orders, navigates by instinct.',
    'druid':       'Forest guardian — patient, speaks in metaphor and natural law.',
    'assassin':    'Precise killer — measures every word, speaks only when needed.',
}

_CHAR_EMOJIS = {
    'wizard': '🧙', 'knight': '⚔️', 'rogue': '🗡️', 'oracle': '🔮',
    'bard': '🎵', 'ranger': '🏹', 'necromancer': '💀', 'paladin': '✨',
    'barbarian': '🪓', 'sea_captain': '⚓', 'druid': '🌿', 'assassin': '🕶️',
    'narrator': '📖',
}

_DEFAULT_LORE = [
    'The Dragon Wars began in the Third Age when the great wyrm Malachar rose from the Ashen Mountains, scorching three kingdoms before being bound by the Conclave.',
    'Wizards of the Silver Tower are bound by an oath never to use fire magic within city walls.',
    'The Knights of the Crimson Order were established to protect the kingdom from demon incursions, each sworn to die before retreat.',
    "Rogues' guilds operate in the shadow markets beneath every major city, trading in secrets as much as stolen goods.",
    'Bards carry the Seal of Free Passage, granting them entry to any tavern or noble court in exchange for songs and news.',
    'The ancient elven script, Quethara, holds the secrets of the first enchantments — written in solidified starlight.',
    'The Moonstone Prophecy foretells a hero rising from the common folk to unite the fractured kingdoms against the coming dark.',
    "Healing potions require dragon's breath moss, only found near volcanic vents in the eastern ranges, harvested at the new moon.",
    'The Underdark is a vast subterranean world populated by creatures that shun daylight and follow their own cruel laws.',
    'Magic is drawn from ley lines that criss-cross the land, converging at ancient standing stones erected by the First People.',
    'The Veil Between Worlds grows thin on the night of the Blood Moon, allowing spirits to pass freely between the living and dead realms.',
    'Sea witches trade in bottled storms and drowned memories; their price is always something you cannot name until it is gone.',
    'The Whispering Wood is forbidden to travelers — those who enter hear voices that slowly replace their own thoughts.',
    "An oracle's prophecy is never wrong, but it is never complete — it always describes the worst possible interpretation of the truth.",
    "The merchant guilds of the Free Cities control every trade route; no caravan moves without their blessing.",
]

_LORE_LABELS = [f'{i+1}. {doc[:65]}…' for i, doc in enumerate(_DEFAULT_LORE)]


# ── Helpers ────────────────────────────────────────────────────────────────

def _msg(role, content):
    """Single Gradio message dict."""
    return {'role': role, 'content': content}


def _make_rag(documents):
    """Build a LoreRAG reusing the notebook's already-loaded _char_encoder."""
    obj = LoreRAG.__new__(LoreRAG)
    obj.documents = documents
    obj.embedder  = _char_encoder
    if documents:
        embs = _char_encoder.encode(documents, convert_to_numpy=True)
        obj.index = faiss.IndexFlatL2(embs.shape[1])
        obj.index.add(embs.astype(np.float32))
    else:
        obj.index = None
    return obj


def _parse_descs(text):
    out = {}
    for line in (text or '').strip().splitlines():
        if ':' in line:
            name, _, desc = line.partition(':')
            name = name.strip().lower().replace(' ', '_')
            if name:
                out[name] = desc.strip()
    return out


def _resolve_lore(labels, custom_text):
    docs = []
    for label in (labels or []):
        try:
            idx = int(label.split('.')[0]) - 1
            if 0 <= idx < len(_DEFAULT_LORE):
                docs.append(_DEFAULT_LORE[idx])
        except (ValueError, IndexError):
            pass
    for line in (custom_text or '').strip().splitlines():
        if line.strip():
            docs.append(line.strip())
    return docs


def _beat_to_msgs(result):
    msgs = [_msg('assistant', f"📖 **NARRATOR**\n{result['narrator_open']}")]
    for name, line in result['characters'].items():
        emoji = _CHAR_EMOJIS.get(name, '👤')
        msgs.append(_msg('assistant', f"{emoji} **{name.upper()}**\n{line}"))
    msgs.append(_msg('assistant', f"📖 **NARRATOR**\n{result['narrator_close']}"))
    return msgs


# ── Event handlers ─────────────────────────────────────────────────────────

def _start_story(selected_chars, custom_descs_text, selected_lore_labels, custom_lore_text):
    if not selected_chars:
        return None, [_msg('assistant', '⚠️ Select at least one character.')]

    overrides   = _parse_descs(custom_descs_text)
    characters  = {
        name: overrides.get(name, _AVAILABLE_ADAPTERS.get(name, f'A {name} character.'))
        for name in selected_chars
    }
    active_lore = _resolve_lore(selected_lore_labels, custom_lore_text)
    story_rag   = _make_rag(active_lore)

    orch = StoryOrchestrator(
        shared=shared,
        tokenizer=tokenizer,
        rag=story_rag,
        lora_dir=LORA_DIR,
        all_texts=all_texts,
        fantasy_base_path=str(fantasy_base_path),
    )
    orch._initialise(premise='', characters=characters)

    cast_lines = '\n'.join(
        f"  {_CHAR_EMOJIS.get(n,'👤')} **{n}** — {d}"
        for n, d in characters.items()
    )
    intro = [_msg('assistant', (
        f"## 🎭 New story started!\n\n**Cast:**\n{cast_lines}\n\n"
        f"📜 {len(active_lore)} lore entries active\n\n"
        "*Enter your first scene beat below.*"
    ))]
    return orch, intro


def _submit_scene(scene_text, orch, chat_history):
    chat_history = list(chat_history or [])
    if orch is None:
        chat_history.append(_msg('assistant', '⚠️ Click **Start / Restart Story** first.'))
        return chat_history, orch, ''
    if not scene_text.strip():
        return chat_history, orch, ''

    chat_history.append(_msg('user', scene_text))
    result = orch.beat(scene_text)
    chat_history.extend(_beat_to_msgs(result))
    return chat_history, orch, ''


def _add_lore(new_entry, current):
    entry = new_entry.strip()
    if not entry:
        return current, ''
    return ((current or '').strip() + '\n' + entry).strip(), ''


print('Gradio helpers loaded.')


Gradio helpers loaded.


In [15]:
# ── Gradio UI ──────────────────────────────────────────────────────────────

with gr.Blocks(title='⚔️ Fantasy Storytelling') as demo:

    _orch_state        = gr.State(None)
    _custom_lore_state = gr.State('')

    gr.Markdown(
        '# ⚔️ Multi-Agent Fantasy Storytelling\n'
        '*Mistral-7B + Hot-Swappable LoRA · RAG lore injection*'
    )

    with gr.Row(equal_height=False):

        # ── LEFT: config ───────────────────────────────────────────────
        with gr.Column(scale=1, min_width=310):

            gr.Markdown('### 🧙 Characters')
            _char_select = gr.CheckboxGroup(
                choices=list(_AVAILABLE_ADAPTERS.keys()),
                value=['wizard', 'rogue', 'necromancer', 'oracle'],
                label='Select characters (2–6)',
            )

            with gr.Accordion('ℹ️ Character descriptions', open=False):
                gr.Markdown(
                    '\n'.join(f'- **{k}** — {v}'
                              for k, v in _AVAILABLE_ADAPTERS.items())
                )

            with gr.Accordion('✏️ Override descriptions (optional)', open=False):
                _custom_descs = gr.Textbox(
                    label='One override per line:  name: description',
                    placeholder=(
                        'wizard: A fallen archmage seeking redemption.\n'
                        'rogue: A spy embedded with the enemy.'
                    ),
                    lines=4,
                )

            gr.Markdown('---\n### 📜 Lore')
            _lore_select = gr.CheckboxGroup(
                choices=_LORE_LABELS,
                value=_LORE_LABELS[:8],
                label='Active lore (top-2 retrieved per prompt via RAG)',
            )

            with gr.Accordion('➕ Add custom lore', open=False):
                _custom_lore_input = gr.Textbox(
                    label='New lore entry',
                    placeholder='The Ashen Catacombs were sealed after…',
                    lines=3,
                )
                _add_lore_btn = gr.Button('Add entry', size='sm')
                _saved_lore_display = gr.Textbox(
                    label='Saved custom lore',
                    lines=3, interactive=False,
                    placeholder='(none yet)',
                )

            gr.Markdown('---')
            _new_story_btn = gr.Button(
                '🎭 Start / Restart Story', variant='primary', size='lg'
            )

        # ── RIGHT: story ───────────────────────────────────────────────
        with gr.Column(scale=2):

            _chatbot = gr.Chatbot(label='Story', height=560)

            with gr.Row():
                _scene_input = gr.Textbox(
                    label='Scene / Next Beat',
                    placeholder='We stand at the entrance to the Ashen Catacombs…',
                    lines=2,
                    scale=5,
                )
                _submit_btn = gr.Button('▶ Submit', variant='primary', scale=1)

            gr.Markdown(
                '_💡 Click **Start Story** first, then submit scene beats. '
                'Press Enter or click Submit._'
            )

    # ── event wiring ──────────────────────────────────────────────────

    _add_lore_btn.click(
        _add_lore,
        inputs=[_custom_lore_input, _custom_lore_state],
        outputs=[_custom_lore_state, _custom_lore_input],
    ).then(
        lambda s: s,
        inputs=[_custom_lore_state],
        outputs=[_saved_lore_display],
    )

    _new_story_btn.click(
        _start_story,
        inputs=[_char_select, _custom_descs, _lore_select, _custom_lore_state],
        outputs=[_orch_state, _chatbot],
    )

    _submit_btn.click(
        _submit_scene,
        inputs=[_scene_input, _orch_state, _chatbot],
        outputs=[_chatbot, _orch_state, _scene_input],
    )

    _scene_input.submit(
        _submit_scene,
        inputs=[_scene_input, _orch_state, _chatbot],
        outputs=[_chatbot, _orch_state, _scene_input],
    )

# ── Launch ─────────────────────────────────────────────────────────────────
# share=True  → public https://xxxx.gradio.live URL (72h)
# share=False → localhost only; SSH tunnel: ssh -L 7860:localhost:7860 user@hpc

demo.queue(max_size=3, default_concurrency_limit=1)
demo.launch(share=True, server_port=7861)


* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://90d89bddc4712fd3a3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[CharacterFactory] Loading TinyLlama director...


Loading weights: 100%|██████████| 201/201 [00:06<00:00, 31.60it/s]


[CharacterFactory] Ready.
[Orchestrator] Using provided cast: ['wizard', 'rogue', 'necromancer', 'oracle']
[Orchestrator] Agent ready: wizard (adapter: wizard)
[Orchestrator] Agent ready: rogue (adapter: rogue)
[Orchestrator] Agent ready: necromancer (adapter: necromancer)
[Orchestrator] Agent ready: oracle (adapter: oracle)
